# 🛒 Proyek Akhir Data Mining
## Preprocessing & Unsupervised Learning — K-Means Clustering

**Dataset**: Mall Customer Segmentation  
**Algoritma**: K-Means Clustering  
**Tools**: Python, Pandas, Scikit-Learn, Matplotlib, Seaborn

## 📦 Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport')

## 1️⃣ Load Dataset Mentah

In [ ]:
df_raw = pd.read_csv('mall_customers_raw.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Cek info dasar
df_raw.info()

In [ ]:
# Statistik deskriptif
df_raw.describe()

## 2️⃣ Cleaning Data

In [ ]:
df = df_raw.copy()

# --- Missing Values ---
print('Missing values per kolom:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Imputasi dengan median
for col in ['Age', 'Annual_Income_k', 'Spending_Score']:
    df[col].fillna(df[col].median(), inplace=True)

print(f'Missing values setelah imputasi: {df.isnull().sum().sum()}')

In [ ]:
# --- Duplikat ---
print(f'Jumlah duplikat: {df.duplicated().sum()}')
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape setelah drop duplikat: {df.shape}')

In [ ]:
# --- Standarisasi Gender ---
print('Sebelum:', df['Gender'].unique())
df['Gender'] = df['Gender'].str.lower().str.strip()
df['Gender'] = df['Gender'].replace({'m': 'male', 'f': 'female'})
print('Sesudah:', df['Gender'].unique())

# Encoding
df['Gender_Enc'] = df['Gender'].map({'male': 0, 'female': 1})
df.head()

## 3️⃣ Scaling / Normalisasi

In [ ]:
features = ['Age', 'Annual_Income_k', 'Spending_Score']
X = df[features].dropna()
df = df.loc[X.index].reset_index(drop=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Statistik setelah StandardScaler:')
pd.DataFrame(X_scaled, columns=features).describe().round(3)

## 4️⃣ Menentukan K Optimal

In [ ]:
inertias, silhouettes = [], []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'K terbaik: {best_k} (Silhouette Score: {max(silhouettes):.4f})')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(k_range), inertias, 'bo-', linewidth=2)
ax1.axvline(best_k, color='red', linestyle='--', label=f'k={best_k}')
ax1.set_title('Elbow Method', fontweight='bold')
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(list(k_range), silhouettes, 'rs-', linewidth=2)
ax2.axvline(best_k, color='blue', linestyle='--', label=f'k={best_k}')
ax2.set_title('Silhouette Score', fontweight='bold')
ax2.set_xlabel('k'); ax2.set_ylabel('Score')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5️⃣ K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

final_sil = silhouette_score(X_scaled, df['Cluster'])
print(f'Silhouette Score: {final_sil:.4f}')
print('\nDistribusi cluster:')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Profil tiap cluster
df.groupby('Cluster')[features].mean().round(2)

## 6️⃣ Visualisasi Hasil Clustering

In [ ]:
palette = sns.color_palette('Set2', best_k)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [
    ('Annual_Income_k', 'Spending_Score', 'Income vs Spending Score'),
    ('Age', 'Spending_Score', 'Age vs Spending Score'),
    ('Age', 'Annual_Income_k', 'Age vs Income'),
]

for ax, (x_col, y_col, title) in zip(axes, pairs):
    for i in range(best_k):
        mask = df['Cluster'] == i
        ax.scatter(df.loc[mask, x_col], df.loc[mask, y_col],
                   c=[palette[i]], label=f'Cluster {i}', s=50, alpha=0.7)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(x_col); ax.set_ylabel(y_col)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D Visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
for i in range(best_k):
    mask = df['Cluster'] == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=[palette[i]], label=f'Cluster {i}', s=60, alpha=0.7)

centers_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1],
            c='black', marker='X', s=200, zorder=5, label='Centroid')

plt.title(f'PCA 2D - Clustering Result (Silhouette={final_sil:.4f})', fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7️⃣ Simpan Hasil

In [ ]:
df.to_csv('mall_customers_with_clusters.csv', index=False)
print('✅ Dataset dengan cluster disimpan!')
print(f'\n📊 Ringkasan:')
print(f'   Dataset awal    : 220 baris')
print(f'   Setelah cleaning: {len(df)} baris')
print(f'   Jumlah cluster  : {best_k}')
print(f'   Silhouette Score: {final_sil:.4f}')

## ✅ Kesimpulan

Preprocessing berhasil membersihkan dataset dari missing values, duplikat, dan inkonsistensi format. K-Means Clustering dengan k=7 menghasilkan segmentasi pelanggan yang bermakna berdasarkan usia, pendapatan, dan skor belanja. Cluster dengan income tinggi dan spending score tinggi merupakan target pemasaran paling potensial.